# In-app feedback: retrieval and analysis

Everything users submit through the two feedback surfaces lands in one table,
`user_feedback`, with a `kind` column telling the two apart:

| `kind` | Written by | Carries |
| --- | --- | --- |
| `source_not_relevant` | The flag button in the Sources table / source dossier | `project_source_snapshot_id` (which source), no `body` |
| `issue_report` | "Report an issue" in the project nav | `body` (the free text) + `page_path` (where they were), no source |

Both rows also carry `project_id`, `user_id` (the Cognito token `sub`, or `dev-user`
locally) and `created_at`.

**Read this before you interpret the flag counts.** The flag is a *toggle*:
un-flagging deletes the row. So `source_not_relevant` rows are **current state, not
history** — you cannot ask "how many sources were ever flagged" or "how often do
people change their mind", only "what is flagged right now". Issue reports are
append-only, so those *are* full history.

If you want flag history, that's a schema change (a `cleared_at` column instead of a
delete, or an `event_log` entry per toggle). Worth deciding before there's much data.

Nothing in the analysis pipeline reads either kind — flagging a source does not
change its status, its selection, or whether it gets cited. These rows exist purely
for us to read.

## 1. Connect

`get_engine()` reads `DATABASE_URL` from the environment, so we load `backend/.env`
first — the same file `make dev` uses. Always print the URL you actually connected
to: the single most common analysis error is reading the wrong database (dev vs
`policy_atlas_test`, which the test suite truncates constantly).

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import select

from policy_atlas.core.db import get_engine
from policy_atlas.core.schema import (
    project,
    project_source_snapshot,
    source_snapshot,
    user_feedback,
)

# Locate the repo from wherever this notebook is opened, rather than assuming a
# fixed depth — this file has already moved once.
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "backend" / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the policy_atlas repo.")
load_dotenv(ROOT / "backend" / ".env")

engine = get_engine()
print("connected to:", engine.url.render_as_string(hide_password=True))

## 2. Load the feedback table

One query for both kinds, with the project name and the source's metadata joined on.
Two things to note in the joins:

- The source joins are **outer** joins — an `issue_report` row has no source, and an
  inner join would silently drop every report.
- The `source_snapshot` join needs an **explicit** `onclause`. `project_source_snapshot`
  has two foreign keys into `source_snapshot` (the abstract envelope and the
  full-text attachment), so SQLAlchemy cannot guess which one you mean.

In [ ]:
feedback_query = (
    select(
        user_feedback.c.user_feedback_id,
        user_feedback.c.created_at,
        user_feedback.c.kind,
        user_feedback.c.user_id,
        user_feedback.c.page_path,
        user_feedback.c.body,
        user_feedback.c.project_id,
        project.c.name.label("project_name"),
        user_feedback.c.project_source_snapshot_id.label("source_id"),
        source_snapshot.c.metadata.label("source_metadata"),
    )
    .select_from(
        user_feedback
        .join(project, user_feedback.c.project_id == project.c.project_id)
        # Outer: issue reports carry no source.
        .outerjoin(
            project_source_snapshot,
            user_feedback.c.project_source_snapshot_id
            == project_source_snapshot.c.project_source_snapshot_id,
        )
        # Explicit onclause — two FKs into source_snapshot make this ambiguous.
        .outerjoin(
            source_snapshot,
            project_source_snapshot.c.source_snapshot_id
            == source_snapshot.c.source_snapshot_id,
        )
    )
    .order_by(user_feedback.c.created_at)
)

feedback = pd.read_sql(feedback_query, engine)

# Titles live in the source_snapshot metadata JSONB, not a column.
feedback["source_title"] = feedback["source_metadata"].map(
    lambda meta: meta.get("title") if isinstance(meta, dict) else None
)
feedback = feedback.drop(columns=["source_metadata"])

print(f"{len(feedback)} feedback rows")
feedback["kind"].value_counts()

## 3. Issue reports

The append-only half. `page_path` is the most actionable column here — it tells you
which screen a complaint came from without the user having to describe it.

In [ ]:
reports = feedback[feedback["kind"] == "issue_report"].copy()

with pd.option_context("display.max_colwidth", 120):
    display(reports[["created_at", "project_name", "page_path", "body"]])

In [ ]:
# Where do complaints come from? Collapse the project uuid out of the path so
# paths from different projects group together.
reports["page"] = reports["page_path"].str.replace(
    r"/projects/[0-9a-f-]{36}", "/projects/<id>", regex=True
)
reports["page"].value_counts()

## 4. Flagged sources, against what the machine decided

This is the interesting join: a flag is a human disagreeing with the pipeline, so it
is only meaningful next to what the pipeline concluded about that source.

`effective_screen_rows()` is the **same helper the API's read models use** to pick
the effective screening verdict per source (newest generation wins, then highest
stage). Join it rather than writing your own "latest screen" logic — a second
implementation will drift from the first.

In [ ]:
from policy_atlas.evidence_base.assess.screen import effective_screen_rows

screens = effective_screen_rows()

flag_query = (
    select(
        user_feedback.c.created_at.label("flagged_at"),
        user_feedback.c.user_id,
        project.c.name.label("project_name"),
        user_feedback.c.project_source_snapshot_id.label("source_id"),
        source_snapshot.c.metadata.label("source_metadata"),
        screens.c.status.label("machine_screen_status"),
        screens.c.screen_stage,
        screens.c.screen_decision_confidence,
        screens.c.screen_basis,
    )
    .select_from(
        user_feedback
        .join(project, user_feedback.c.project_id == project.c.project_id)
        .join(
            project_source_snapshot,
            user_feedback.c.project_source_snapshot_id
            == project_source_snapshot.c.project_source_snapshot_id,
        )
        .join(
            source_snapshot,
            project_source_snapshot.c.source_snapshot_id
            == source_snapshot.c.source_snapshot_id,
        )
        # Outer: a source can be flagged before it has been screened at all.
        .outerjoin(
            screens,
            screens.c.project_source_snapshot_id
            == user_feedback.c.project_source_snapshot_id,
        )
    )
    .where(user_feedback.c.kind == "source_not_relevant")
    .order_by(user_feedback.c.created_at)
)

flags = pd.read_sql(flag_query, engine)
flags["source_title"] = flags["source_metadata"].map(
    lambda meta: meta.get("title") if isinstance(meta, dict) else None
)
flags = flags.drop(columns=["source_metadata"])

print(f"{len(flags)} sources currently flagged as not relevant")
flags

### Where the disagreement is

`machine_screen_status` is what screening decided; the flag is the human saying "not
relevant". So:

- `relevant` → **a real disagreement.** The pipeline let this source through and a
  human rejected it. These are the rows worth reading — they're screening
  false positives, and `screen_reason` in the event log will tell you why it passed.
- `not_relevant` → **agreement.** The pipeline already screened it out; the human
  is confirming. Useful as a sanity check on the flag itself.
- `NaN` → the source was never screened (flagged straight from the found list).

A high disagreement rate at high `screen_decision_confidence` is the signal that
matters: the screening prompt is confidently wrong about something.

In [ ]:
if flags.empty:
    print("No flags recorded yet — flag a source in the Sources tab and re-run.")
else:
    summary = (
        flags.assign(machine=flags["machine_screen_status"].fillna("never screened"))
        .groupby("machine")
        .agg(
            flagged=("source_id", "count"),
            mean_confidence=("screen_decision_confidence", "mean"),
        )
        .sort_values("flagged", ascending=False)
    )
    display(summary)

    disagreements = flags[flags["machine_screen_status"] == "relevant"]
    print(f"\n{len(disagreements)} of {len(flags)} flags contradict the pipeline")
    with pd.option_context("display.max_colwidth", 90):
        display(
            disagreements[
                ["project_name", "source_title", "screen_decision_confidence", "screen_basis"]
            ]
        )

## 5. Volume over time

Both kinds together. With a handful of rows this is noise; it becomes useful once
there are weeks of data. `created_at` is timezone-aware UTC — resample on it
directly rather than stripping the timezone.

In [ ]:
by_day = (
    feedback.set_index("created_at")
    .groupby("kind")
    .resample("1D")
    .size()
    .unstack("kind", fill_value=0)
)
display(by_day)

# Who is submitting? Locally this is always `dev-user`; in production it is the
# Cognito `sub`, which is an opaque uuid — not an email address.
display(feedback.groupby(["user_id", "kind"]).size().unstack(fill_value=0))

## 6. Doing this against production

The queries above are unchanged. What changes is **getting a connection**, because
Aurora sits in private subnets with no public endpoint — there is nothing to point
`DATABASE_URL` at from a laptop until you open a tunnel.

Four differences to plan for:

**1. Network.** You need an SSM port-forward through the fck-nat instance (no
bastion, no inbound ports, IAM-gated). The verbatim recipe is in
`infra/DEPLOYMENT.md` § 6 "Developer DB access" — follow that, not a copy here that
can drift. Prereqs: AWS CLI, `session-manager-plugin`, and IAM allowing
`ssm:StartSession` on that instance. Once the tunnel is up, Aurora is on
`localhost:15432`.

**2. Credentials.** There is no `.env`. The password lives in the Secrets Manager
secret whose name is in SSM at `/policy_atlas_v3/db/secret_name`; the secret's
`db_connection_string` field is the same value the ECS task gets as `DATABASE_URL`.
Read it at runtime — never paste it into a notebook cell, because notebook outputs
are saved to disk and committed.

**3. The database is named differently.** Local docker-compose is `policy_atlas`;
Aurora's is `policy_atlas_db`. And production requires TLS, so the URL needs
`?sslmode=require`. A URL copied from local will fail on both counts.

**4. Read-only discipline.** This is live user data with no undo. Nothing in this
notebook writes, but one careless cell could — so connect with an engine you cannot
accidentally commit through. `engine.connect()` in SQLAlchemy 2 does not
autocommit, and `pd.read_sql` never writes, so passing a *connection* rather than
the engine and never calling `.commit()` is enough.

One hard warning from `DEPLOYMENT.md`: a tunnel for psql/pandas inspection is always
fine, but do **not** boot a local API against Aurora. It becomes a second instance
sharing the database, and its orphan sweep will interrupt the staging service's
running walks.

In [ ]:
# Production connection — run the tunnel from infra/DEPLOYMENT.md § 6 first, then:
#
# import boto3, json
# ssm = boto3.client("ssm", region_name="eu-west-2")
# secret_name = ssm.get_parameter(Name="/policy_atlas_v3/db/secret_name")["Parameter"]["Value"]
# creds = json.loads(
#     boto3.client("secretsmanager", region_name="eu-west-2")
#     .get_secret_value(SecretId=secret_name)["SecretString"]
# )
#
# # Through the tunnel: localhost:15432, the Aurora database name, TLS required.
# prod_url = (
#     f"postgresql+psycopg://{creds['username']}:{creds['password']}"
#     "@localhost:15432/policy_atlas_db?sslmode=require"
# )
# prod_engine = create_engine(prod_url)
#
# # Read through a connection you never commit — pd.read_sql cannot write, and an
# # uncommitted transaction rolls back when the block exits.
# with prod_engine.connect() as conn:
#     prod_feedback = pd.read_sql(feedback_query, conn)
#
# print(len(prod_feedback), "rows")   # never print prod_url or creds

## Housekeeping

`scripts/scratchpad/` is **not** in `.gitignore`, and this notebook saves its
outputs — which already include real feedback text and project names. A
`git add -A` would commit them. Either add `scripts/scratchpad/` to
`.gitignore`, or clear all outputs before committing. That matters more once
you have run it against production, where the text is other people's.

The kernel must be the backend virtualenv (`backend/.venv`, shown as
`policy-atlas`), which is what makes `import policy_atlas` work from this
directory — the import resolves through the installed package, not the cwd.

`pandas` is installed in that venv but is **not** declared in
`backend/pyproject.toml`, so a `uv sync` will remove it. Declaring it is a
dependency change, which `AGENTS.md` gates on approval — until then,
`uv add --dev pandas` (with that approval) or `uv run --with pandas` keeps it
available.